In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import librosa
import json
import torch
import math
import soundfile as sf

In [2]:
audiocaps_wav_path = "/blob/v-yuancwang/TTADataset/processed_data"
audiocaps_info_path = "/blob/v-yuancwang/TTADataset/processed_data/total_train_filter_speech.json"
with open(audiocaps_info_path, "r") as f:
    audiocaps_info = json.load(f)
traffic_info = []
children_info = []
beach_info = []
raining_info = []
gym_info = []
subway_info = []
shopping_info = []
bell_info = []
traffic_key_words = ["traffic", "car", "bus", "motorcycle", "bicycle", "train"]
children_key_words = ["children", "child", "kid", "baby", "infant", "kids"]
beach_key_words = ["ocean wave", "ocean waving", "ocean waves", "ocean"]
raining_key_words = ["raining", "thundering", "storm", "thunder", "rain and thunder"]
gym_key_words = ["gym", "sports center", "sport center", "sports", "basketball", "Basketball"]
subway_key_words = ["subway", "bus", "underground", "tube", "subway station", "underground station", "tube station"]
shopping_key_words = ["shopping", "mall", "supermarket", "store", "shop", "shopping mall", "supermarket", "grocery store"]
bell_key_words = ["bell", "church bell", "church bells", "bells", "church", "clock alarm", "alarm clock", "clock", "alarms"]
for i in range(len(audiocaps_info)):
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in traffic_key_words]):
        traffic_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in children_key_words]):
        children_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in beach_key_words]):
        beach_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in raining_key_words]):
        raining_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in gym_key_words]):
        gym_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in subway_key_words]):
        subway_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in shopping_key_words]):
        shopping_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in bell_key_words]):
        bell_info.append(audiocaps_info[i])
print("traffic_info:", len(traffic_info))
print("children_info:", len(children_info))
print("beach_info:", len(beach_info))
print("raining_info:", len(raining_info))
print("gym_info:", len(gym_info))
print("subway_info:", len(subway_info))
print("shopping_info:", len(shopping_info))
print("bell_info:", len(bell_info))

traffic_info: 21236
children_info: 5378
beach_info: 1097
raining_info: 1692
gym_info: 1069
subway_info: 3371
shopping_info: 1217
bell_info: 7180


In [3]:
def get_noise_scale(wavform, noise, snr_min=5, snr_max=10):
    wavform_power = np.sum(wavform**2) / len(wavform)
    noise_power = np.sum(noise**2) / len(noise)
    snr = np.random.rand() * (snr_max - snr_min) + snr_min
    noise_scale = np.sqrt(wavform_power / (noise_power * 10 ** (snr / 10)))
    return noise_scale, snr


def calculate_snr(wavform, noise):
    wavform_power = np.sum(wavform**2) / len(wavform)
    noise_power = np.sum(noise**2) / len(noise)
    snr = 10 * math.log10(wavform_power / noise_power)
    return snr

def gen_env_speech(speech, env, snr_min, snr_max):
    noise_scale, snr = get_noise_scale(speech, env, snr_min, snr_max)
    return speech + env * noise_scale

In [4]:
# speech_data_path = "/home/t-zeqianju/yuancwang/temp_speech_env/speech"
# mix_data_path = "/home/t-zeqianju/yuancwang/temp_speech_env/mix"
# test_json_path = "/home/t-zeqianju/yuancwang/Amphion/temp_meta_info/test_temp.json"
speech_data_path = "/home/t-zeqianju/yuancwang/LibriSpeech/train-clean-100"
mix_data_path = "/home/t-zeqianju/yuancwang/temp_speech_env/train_new/mix"
test_json_path = "/home/t-zeqianju/yuancwang/temp_speech_env/result_0524.json"

In [5]:
with open(test_json_path, "r", encoding="utf_8_sig") as f:
    test_info = json.load(f)
beach_sum = 0
traffic_sum = 0
children_sum = 0
raining_sum = 0
gym_sum = 0
subway_sum = 0
shopping_sum = 0
bell_sum = 0
print(len(test_info))
for i in range(len(test_info)):
    if test_info[i]["background"] == "at the sea beach":
        beach_sum += 1
    if test_info[i]["background"] == "driving or traffic":
        traffic_sum += 1
    if test_info[i]["background"] == "children's voice":
        children_sum += 1
    if test_info[i]["background"] == "raining and thundering":
        raining_sum += 1
    if test_info[i]["background"] == "at the sports center":
        gym_sum += 1
    if test_info[i]["background"] == "on the bus or subway":
        subway_sum += 1
    if test_info[i]["background"] == "in the shopping center":
        shopping_sum += 1
    if test_info[i]["background"] == "alarms or bells":
        bell_sum += 1

print("beach_sum:", beach_sum)
print("traffic_sum:", traffic_sum)
print("children_sum:", children_sum)
print("raining_sum:", raining_sum)
print("gym_sum:", gym_sum)
print("subway_sum:", subway_sum)
print("shopping_sum:", shopping_sum)
print("bell_sum:", bell_sum)

7184
beach_sum: 898
traffic_sum: 898
children_sum: 898
raining_sum: 898
gym_sum: 898
subway_sum: 898
shopping_sum: 898
bell_sum: 898


In [6]:
beach_sum = 0
traffic_sum = 0
children_sum = 0
raining_sum = 0
gym_sum = 0
subway_sum = 0
shopping_sum = 0
bell_sum = 0

from tqdm import tqdm

saved_path = os.listdir("/home/t-zeqianju/yuancwang/temp_speech_env/train_new/mix")
print(len(saved_path))
print(saved_path)

with open(test_json_path, "r", encoding="utf_8_sig") as f:
    test_info = json.load(f)
for i in tqdm(range(len(test_info))):

    background = test_info[i]["background"]
    uid = test_info[i]["uid"]

    if background == "driving or traffic":
        mix_path = os.path.join(mix_data_path, uid + "-traffic" + ".wav")
    elif background == "children's voice":
        mix_path = os.path.join(mix_data_path, uid + "-children" + ".wav")
    elif background == "at the sea beach":
        mix_path = os.path.join(mix_data_path, uid + "-beach" + ".wav")
    elif background == "raining and thundering":
        mix_path = os.path.join(mix_data_path, uid + "-raining" + ".wav")
    elif background == "at the sports center":
        mix_path = os.path.join(mix_data_path, uid + "-gym" + ".wav")
    elif background == "on the bus or subway":
        mix_path = os.path.join(mix_data_path, uid + "-subway" + ".wav")
    elif background == "in the shopping center":
        mix_path = os.path.join(mix_data_path, uid + "-shopping" + ".wav")
    elif background == "alarms or bells":
        mix_path = os.path.join(mix_data_path, uid + "-bell" + ".wav")

    # if uid + "-bell" + ".wav" in saved_path:
    #     # print(mix_path)
    #     # print("continue")
    #     continue

    # choice env audio
    if background == "driving or traffic":
        env_info = traffic_info
        traffic_sum += 1
    elif background == "children's voice":
        env_info = children_info
        children_sum += 1
    elif background == "at the sea beach":
        env_info = beach_info
        beach_sum += 1
    elif background == "raining and thundering":
        env_info = raining_info
        raining_sum += 1
    elif background == "at the sports center":
        env_info = gym_info
        gym_sum += 1
    elif background == "on the bus or subway":
        env_info = subway_info
        subway_sum += 1
    elif background == "in the shopping center":
        env_info = shopping_info
        shopping_sum += 1
    elif background == "alarms or bells":
        env_info = bell_info
        bell_sum += 1

    # random_idx = np.random.randint(0, len(env_info))
    # random_env_uid = env_info[random_idx]["Uid"]
    # random_env_dataset = env_info[random_idx]["Dataset"]
    # random_env_caption = env_info[random_idx]["Caption"]
    # random_env_path = os.path.join(audiocaps_wav_path, random_env_dataset, "wav", random_env_uid + ".wav")
    # # load env audio
    # env_audio = np.array([0.])
    # while sum(env_audio) == 0:
    #     env_audio, sr = librosa.load(random_env_path, sr=16000)
    #     # load speech audio
    #     # speech_path = os.path.join(speech_data_path, uid + ".wav")
    #     spk = uid.split("-")[0]
    #     chapter = uid.split("-")[1]
    #     speech_path = os.path.join(speech_data_path, spk, chapter, uid + ".flac")
    #     speech_audio, sr = librosa.load(speech_path, sr=16000)
    #     speech_len = len(speech_audio)
    #     env_len = len(env_audio)
    #     # padding env audio if env_len < speech_len
    #     if env_len < speech_len:
    #         env_audio = np.pad(env_audio, (0, speech_len - env_len), mode="wrap")
    #     if env_len > speech_len:
    #         env_audio_start = np.random.randint(0, env_len - speech_len)
    #     else:
    #         env_audio_start = 0
    #     env_audio = env_audio[env_audio_start:env_audio_start + speech_len]
    # # print(len(speech_audio), len(env_audio))
    # # mix audio
    # mix_audio = gen_env_speech(speech_audio, env_audio, 5, 15)
    # # save mix audio
    # if background == "driving or traffic":
    #     mix_path = os.path.join(mix_data_path, uid + "-traffic" + ".wav")
    # elif background == "children's voice":
    #     mix_path = os.path.join(mix_data_path, uid + "-children" + ".wav")
    # elif background == "at the sea beach":
    #     mix_path = os.path.join(mix_data_path, uid + "-beach" + ".wav")
    # elif background == "raining and thundering":
    #     mix_path = os.path.join(mix_data_path, uid + "-raining" + ".wav")
    # elif background == "at the sports center":
    #     mix_path = os.path.join(mix_data_path, uid + "-gym" + ".wav")
    # elif background == "on the bus or subway":
    #     mix_path = os.path.join(mix_data_path, uid + "-subway" + ".wav")
    # elif background == "in the shopping center":
    #     mix_path = os.path.join(mix_data_path, uid + "-shopping" + ".wav")
    # elif background == "alarms or bells":
    #     mix_path = os.path.join(mix_data_path, uid + "-bell" + ".wav")
    # sf.write(mix_path, mix_audio, sr)

print("beach_sum:", beach_sum)
print("traffic_sum:", traffic_sum)
print("children_sum:", children_sum)
print("raining_sum:", raining_sum)
print("gym_sum:", gym_sum)
print("subway_sum:", subway_sum)
print("shopping_sum:", shopping_sum)
print("bell_sum:", bell_sum)

7184
['26-495-0075-subway.wav', '2002-139469-0069-subway.wav', '412-126975-0081-raining.wav', '4297-13009-0013-bell.wav', '8095-274346-0055-children.wav', '1040-133433-0061-bell.wav', '1355-39947-0070-traffic.wav', '4051-11217-0003-traffic.wav', '6880-216547-0062-bell.wav', '89-219-0052-gym.wav', '6081-42010-0018-children.wav', '8419-286667-0012-shopping.wav', '3526-175658-0005-subway.wav', '1867-154075-0038-bell.wav', '6529-62554-0040-raining.wav', '4898-28461-0032-subway.wav', '4397-15666-0013-children.wav', '3214-167606-0024-raining.wav', '1963-142393-0038-raining.wav', '1723-141149-0076-traffic.wav', '2518-154825-0017-children.wav', '2910-131096-0032-traffic.wav', '39-121915-0031-raining.wav', '2007-132570-0016-raining.wav', '6531-61334-0092-bell.wav', '6437-66172-0022-gym.wav', '2893-139322-0034-shopping.wav', '6415-116629-0015-shopping.wav', '669-129061-0075-gym.wav', '1081-125237-0092-subway.wav', '1116-132847-0029-children.wav', '625-132118-0037-raining.wav', '7059-77900-0013-s

100%|██████████| 7184/7184 [00:00<00:00, 543278.94it/s]

beach_sum: 898
traffic_sum: 898
children_sum: 898
raining_sum: 898
gym_sum: 898
subway_sum: 898
shopping_sum: 898
bell_sum: 898
